# 01 - Exploratory data analysis

Goal: understand the class imbalance, the Amount and Time columns, and which features
separate fraud from normal transactions. Figures are saved in `reports/figures/`.

In [ ]:
import os
# the notebooks live in notebooks/, but all paths below are from the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

os.makedirs("reports/figures", exist_ok=True)
df = pd.read_csv("data/creditcard.csv")
sns.set_style("whitegrid")
df.head()

## Data quality and duplicates

In [ ]:
print(df.dtypes.value_counts())
print(df.describe().T[["mean", "std", "min", "max"]].round(3))

dups = df[df.duplicated(keep="first")]
print("\nDuplicate rows:", len(dups))
print("Duplicates by class:\n", dups["Class"].value_counts())
print("Fraud rows before/after dedup:", df["Class"].sum(), df.drop_duplicates()["Class"].sum())

No missing values and everything is numeric. There are 1,081 duplicates (1,062 legit and 19 fraud).
Dropping them removes 19 fraud rows (about 3.9% of fraud), which is a bigger share than for
legit rows (about 0.4%), so leaving them in would inflate the test scores. They are removed in preprocessing.

## Class imbalance

In [ ]:
counts = df["Class"].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax[0])
ax[0].set_title("Class counts (linear)")
sns.barplot(x=counts.index, y=counts.values, ax=ax[1]).set_yscale("log")
ax[1].set_title("Class counts (log scale)")
plt.tight_layout(); plt.savefig("reports/figures/class_imbalance.png", dpi=150); plt.show()
print("Imbalance ratio: 1 fraud per", round(counts[0] / counts[1]), "legit")

About 1 fraud for every 578 normal transactions. A model that always says "legit" would get
99.83% accuracy and catch nothing, so accuracy is useless here. I use PR-AUC, precision, recall
and F1 instead.

## Transaction amount

In [ ]:
print(df.groupby("Class")["Amount"].describe().round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Class", y="Amount", ax=ax[0]).set_yscale("log")
ax[0].set_title("Amount by class (log)")
sns.histplot(data=df, x=np.log1p(df["Amount"]), hue="Class",
             stat="density", common_norm=False, bins=50, ax=ax[1])
ax[1].set_title("log(1+Amount) density by class")
plt.tight_layout(); plt.savefig("reports/figures/amount_by_class.png", dpi=150); plt.show()

Fraud has a higher mean (122 vs 88) but a much lower median (9.25 vs 22.00), so many frauds are very
small transactions (the 25th percentile is 1.00) while a few large ones pull the mean up. The two
distributions overlap a lot, so Amount alone cannot detect fraud. Because it is so skewed I use
`log1p` and then standard scaling.

## Time

In [ ]:
df["Hour"] = (df["Time"] // 3600) % 24
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=df, x="Hour", hue="Class", stat="density", common_norm=False, bins=24, ax=ax)
ax.set_title("Transactions by hour (data covers about 2 days)")
plt.tight_layout(); plt.savefig("reports/figures/time_by_class.png", dpi=150); plt.show()

`Time` is seconds since the first transaction (the data covers about 48 hours), not a real clock time, so the
hour plot is only a rough hint. A new transaction in the API would not have a meaningful `Time`, so I drop it.

## Which features separate fraud?

In [ ]:
corr = df.drop(columns="Hour").corr()["Class"].drop("Class").sort_values()
print(corr.head(6)); print(corr.tail(6))

top = corr.abs().sort_values(ascending=False).head(6).index
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for a, c in zip(axes.ravel(), top):
    sns.kdeplot(data=df, x=c, hue="Class", common_norm=False, ax=a)
    a.set_title(c)
plt.tight_layout(); plt.savefig("reports/figures/top_features.png", dpi=150); plt.show()

Strongest negative correlations with fraud: V17 (-0.33), V14 (-0.30), V12 (-0.26), V10 (-0.22), V16, V3.
Strongest positive: V11 (0.15), V4 (0.13), V2 (0.09).

In the density plots legit transactions form a tight spike around 0 while fraud is wider and shifted to
negative values with long tails. Note that the densities are normalised per class, so the plot shows the
shape and not how many transactions there are. The extreme fraud values are the signal, so I do not clip outliers.
Some plots (V17, V12, V10) have bumps, which hints at more than one type of fraud and is one reason to
try non-linear models.

## Scaling check

In [ ]:
vcols = [f"V{i}" for i in range(1, 29)]
print(df[["Time", "Amount"]].describe().round(2))
print("V-feature std range:", df[vcols].std().min().round(2), "-", df[vcols].std().max().round(2))

## Key observations

1. Severe imbalance: 0.173% fraud (0.167% after removing duplicates).
2. 1,081 duplicate rows (19 fraud), removed to prevent leakage.
3. No missing values, all features numeric.
4. Fraud amounts are skewed small (median 9.25 vs 22.00) but overlap a lot with legit ones.
5. V17, V14, V12 and V10 are the most discriminative features.
6. `Time` is elapsed seconds over 2 days, not useful for new transactions.
7. V1-V28 are anonymised PCA components, so results cannot be explained in business terms (a limitation).
8. `Amount` needs scaling, V1-V28 are already on similar scales.